# Does the marked verb describe vertical movement?

Run each cell from the top. You will change a reading rule, train a classifier, and read a disagreement. The four provisional labels come from Qwen3.5-9B: movement happens (CT+), is negated (CT−, written CT-), is left open (Uu), or the verb has another sense (NM).

### Load the class data
Run this setup cell once. It downloads the passages, provisional labels, and saved embeddings from the course site and aligns them by passage ID.

In [ ]:
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

url = "https://nlp.lang.osaka-u.ac.jp/dh/b/materials/movement-classification.zip"
archive, _ = urlretrieve(url)
with ZipFile(archive) as bundle:
    passages = pd.read_json(bundle.open("class7/passages.jsonl"), lines=True).set_index("id")
    labels = pd.read_json(bundle.open("class7/llm-labels.jsonl"), lines=True).set_index("id")
    with np.load(bundle.open("class7/embeddings.npz")) as saved:
        vectors = pd.DataFrame(saved["vectors"].astype("float32"), index=saved["ids"])

passages["reference"] = labels["label"]

### 1. Read before predicting
Read the marked verb and the neighboring sentences. What falls? Choose a label before continuing.

In [ ]:
passage = passages.loc["dev-0054"]
print(passage["previous"])
print(passage["left"] + "【" + passage["target"] + "】" + passage["right"])
print(passage["next"])

This rule predicts NM when the words immediately before the verb end with the cue. Otherwise it predicts CT+. Predict the result, then change the cue to `声が` and rerun the cell. Which reading can the new cue still miss?

In [ ]:
cue = "眠りに"
if passage["left"].endswith(cue):
    prediction = "NM"
else:
    prediction = "CT+"
print(prediction, passage["reference"])

### 2. Fit on training works
Each row of `vectors` represents one marked passage. The classifier learns from 420 training passages. The development and test passages come from other works.

In [ ]:
train = passages[passages["split"] == "train"]
dev = passages[passages["split"] == "dev"].dropna(subset=["reference"])
test = passages[passages["split"] == "test"]

train_vectors = vectors.loc[train.index]
train_labels = train["reference"]
model = LogisticRegression(max_iter=2000)
model.fit(train_vectors, train_labels)

Check agreement on the 59 development passages with a readable label. Each row is a reference label and each column is a prediction. How many Uu passages did the classifier recover? Write your answer before opening the test results.

In [ ]:
dev_predictions = model.predict(vectors.loc[dev.index])
pd.crosstab(dev["reference"], dev_predictions, colnames=["prediction"])

### 3. Inspect a disagreement on held-out works
Count the agreements, then inspect the first disagreement. Which words support your reading? Change `0` to `1` to read another case.

In [ ]:
test_predictions = model.predict(vectors.loc[test.index])
results = test.assign(prediction=test_predictions)
agrees = results["prediction"] == results["reference"]
print(agrees.sum(), "/", len(results))
disagreements = results[~agrees]

In [ ]:
case = disagreements.iloc[0]
print(case["title"], case["reference"], case["prediction"])
print(case["previous"])
print(case["left"] + "【" + case["target"] + "】" + case["right"])
print(case["next"])